# Sentiment Classification of YouTube Comments on IHSG
### Stacking Ensemble: Naive Bayes + SVM + BiLSTM -> 1 final output

Notebook flow:
1. Configuration (API keys and the **YouTube video list variable**)
2. Fetch **top-level comments only** (replies are excluded) via the YouTube Data API v3
3. Labeling (manual)
4. Text preprocessing
5. Base models: Naive Bayes, SVM, BiLSTM
6. Stacking (out-of-fold predictions + Logistic Regression meta-learner)
7. Evaluation and prediction on new comments

## 1. Configuration

In [9]:
import os

# ====== FILL IN THIS SECTION ======
API_KEY = os.getenv("YOUTUBE_API_KEY")

# Video list variable: full URLs, short URLs (youtu.be) or raw video IDs are all accepted
VIDEO_LIST = [
    "https://youtu.be/oL8nCaXXJPE?si=NFrPfKvdYIEldSrm",
    "https://youtube.com/shorts/kuCxBDAUZz4?si=HRv7C2bGES2tnDrS",
    "https://youtube.com/shorts/svYRuI09D9U?si=8xP_Hi6i5fx9zX94",
    "https://youtu.be/I1NBxuJ0hM0?si=XgE8aZoYx3Nxq8BJ"
]

MAX_COMMENTS_PER_VIDEO = 1000   # max top-level comments per video
ORDER = "relevance"             # "relevance" or "time"

# Preprocessing / model
USE_STEMMING = True   # Sastrawi stemming (more accurate but slow)
SEED = 42
N_FOLDS = 5
MAX_WORDS = 20000
MAX_LEN = 60
EPOCHS = 15
BATCH_SIZE = 32

## 2. Fetch YouTube comments (top-level comments only)

Note: `commentThreads.list` with `part="snippet"` returns only the main comment
(`topLevelComment`). Replies are not requested (`replies` is not included) and are not stored.

In [12]:
import re, time
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

youtube = build("youtube", "v3", developerKey=API_KEY)

def extract_video_id(s):
    s = s.strip()
    patterns = [
        r"(?:v=)([A-Za-z0-9_-]{11})",
        r"youtu\.be/([A-Za-z0-9_-]{11})",
        r"youtube\.com/shorts/([A-Za-z0-9_-]{11})",
        r"youtube\.com/embed/([A-Za-z0-9_-]{11})",
    ]
    for p in patterns:
        m = re.search(p, s)
        if m:
            return m.group(1)
    if re.fullmatch(r"[A-Za-z0-9_-]{11}", s):
        return s
    return None

def fetch_top_level_comments(video_id, max_comments=1000, order="relevance"):
    rows, token = [], None
    while len(rows) < max_comments:
        try:
            resp = youtube.commentThreads().list(
                part="snippet",              # snippet only -> no replies
                videoId=video_id,
                maxResults=100,
                pageToken=token,
                order=order,
                textFormat="plainText",
            ).execute()
        except HttpError as e:
            print(f"[!] {video_id}: {e}")
            break
        for item in resp.get("items", []):
            sn = item["snippet"]["topLevelComment"]["snippet"]
            rows.append({
                "video_id": video_id,
                "comment_id": item["snippet"]["topLevelComment"]["id"],
                "text": sn["textDisplay"],
                "like_count": sn.get("likeCount", 0),
                "published_at": sn["publishedAt"],
            })
            if len(rows) >= max_comments:
                break
        token = resp.get("nextPageToken")
        if not token:
            break
        time.sleep(0.2)
    return rows

all_rows = []
for v in VIDEO_LIST:
    vid = extract_video_id(v)
    if vid is None:
        print(f"[!] Could not parse a video ID from: {v}")
        continue
    got = fetch_top_level_comments(vid, MAX_COMMENTS_PER_VIDEO, ORDER)
    print(f"{vid}: {len(got)} top-level comments")
    all_rows.extend(got)

df_raw = pd.DataFrame(all_rows)
if len(df_raw):
    df_raw = df_raw.drop_duplicates(subset="comment_id").drop_duplicates(subset="text").reset_index(drop=True)
print("Total unique comments:", len(df_raw))
df_raw.head()

oL8nCaXXJPE: 913 top-level comments
kuCxBDAUZz4: 1000 top-level comments
svYRuI09D9U: 932 top-level comments
I1NBxuJ0hM0: 920 top-level comments
Total unique comments: 3742


,video_id,comment_id,text,like_count,published_at
0,oL8nCaXXJPE,UgxopU9S_9G0VE8E29Z4AaABAg,msci cuma minta nasi goreng. indonesia ngasihn...,593,2026-10-16T11:54:44Z
1,oL8nCaXXJPE,UgzmYS3_2HV6RSRVFcx4AaABAg,"Wajar saja di turunkan, memang negara terlalu ...",326,2026-10-16T15:15:01Z
2,oL8nCaXXJPE,UgyBfgi0k0hVc63DFjh4AaABAg,Selamat ye bennix Indonesia dah jadi maju meng...,15,2026-10-18T16:37:29Z
3,oL8nCaXXJPE,UgxIPVMKRWsE9CGtN394AaABAg,"Watch from malaysia, hi apa khabar",22,2026-10-18T19:21:24Z
4,oL8nCaXXJPE,UgwBV5TrPY7LlSxBBQV4AaABAg,"Sampaikan kepada pak Prabowo, jumlah korban ka...",287,2026-10-16T16:05:02Z


In [13]:
# Save the scraped comments and a template file for manual labeling
# Line breaks (Enter) inside a comment are turned into a literal "\\n" escape so every
# comment stays on ONE line in the CSV (much tidier when opened in Excel / a text editor).
def escape_newlines(t):
    t = str(t).replace("\r\n", "\n").replace("\r", "\n")   # normalize all line breaks
    t = t.replace("\t", " ")                                 # tabs -> space
    t = re.sub(r"\n+", r"\\n", t)                            # one or more Enters -> literal \n
    return t.strip()

df_save = df_raw.copy()
df_save["text"] = df_save["text"].apply(escape_newlines)
df_save.to_csv("comments_raw.csv", index=False, encoding="utf-8-sig")

label_template = df_save[["comment_id", "text"]].copy()
label_template["label"] = ""   # fill in manually: positive / neutral / negative
label_template.to_csv("comments_to_label.csv", index=False, encoding="utf-8-sig")
print("Saved: comments_raw.csv and comments_to_label.csv (newlines escaped as \\n)")

Saved: comments_raw.csv and comments_to_label.csv (newlines escaped as \n)


## 4. Preprocessing

In [ ]:
import emoji
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Normalization of Indonesian slang/abbreviations (the comments are in Indonesian)
SLANG = {
    "gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak", "ngga": "tidak", "tdk": "tidak", "g": "tidak",
    "yg": "yang", "dgn": "dengan", "utk": "untuk", "krn": "karena", "bgt": "banget", "sm": "sama",
    "udh": "sudah", "udah": "sudah", "blm": "belum", "jgn": "jangan", "tp": "tapi", "klo": "kalau", "kalo": "kalau",
    "gw": "saya", "gue": "saya", "lu": "kamu", "lo": "kamu", "emg": "memang", "bs": "bisa",
    "cuan": "untung", "boncos": "rugi", "nyangkut": "rugi", "gorengan": "manipulasi",
    "ihsg": "ihsg", "ara": "naik", "arb": "turun",
}

# Keep negation words because they matter for sentiment
stop_words = set(StopWordRemoverFactory().get_stop_words())
stop_words -= {"tidak", "bukan", "belum", "jangan", "tanpa", "kurang"}

stemmer = StemmerFactory().create_stemmer() if USE_STEMMING else None

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    t = re.sub(r"@\w+", " ", t)
    t = re.sub(r"#(\w+)", r"\1", t)
    try:
        t = emoji.demojize(t, delimiters=(" ", " "), language="id")
    except Exception:
        t = emoji.replace_emoji(t, replace=" ")
    t = t.replace("_", " ")
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)          # collapse repeated characters: "mantaaap" -> "mantaap"
    t = re.sub(r"[^a-z\s]", " ", t)
    toks = [SLANG.get(w, w) for w in t.split()]
    toks = [w for w in toks if w not in stop_words and len(w) > 1]
    if stemmer:
        toks = [stemmer.stem(w) for w in toks]
    return " ".join(toks)

df = df_lab.copy()
df["clean"] = df["text"].apply(clean_text)
df = df[df["clean"].str.strip() != ""].reset_index(drop=True)
print("Number of samples after preprocessing:", len(df))
df[["text", "clean", "label"]].head(10)

## 5. Train/test split and label encoding

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df["label"])
K = len(le.classes_)
print("Classes:", list(le.classes_))
assert df["label"].value_counts().min() >= max(N_FOLDS, 5), "A class has too little data for cross-validation."

X_train, X_test, y_train, y_test = train_test_split(
    df["clean"].values, y, test_size=0.2, stratify=y, random_state=SEED
)
print("Train:", len(X_train), "| Test:", len(X_test))

## 6. Base model

- **Naive Bayes** (TF-IDF unigram+bigram)
- **SVM** (TF-IDF, LinearSVC + calibration to output probabilities)
- **BiLSTM** (embedding trained from scratch)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(SEED); tf.random.set_seed(SEED)

# ---------- Naive Bayes ----------
def make_nb():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf", MultinomialNB(alpha=0.5)),
    ])

# ---------- SVM ----------
def make_svm():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf", CalibratedClassifierCV(LinearSVC(class_weight="balanced"), cv=3)),
    ])

# ---------- BiLSTM ----------
tok = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tok.fit_on_texts(X_train)   # fit the tokenizer on training data only

def to_seq(texts):
    return pad_sequences(tok.texts_to_sequences(texts), maxlen=MAX_LEN, padding="post", truncating="post")

def build_bilstm(n_classes):
    m = Sequential([
        Embedding(MAX_WORDS, 128, mask_zero=True),
        Bidirectional(LSTM(64)),
        Dropout(0.5),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(n_classes, activation="softmax"),
    ])
    m.compile(loss="sparse_categorical_crossentropy", optimizer=Adam(1e-3), metrics=["accuracy"])
    return m

def train_bilstm(texts, labels):
    Xs = to_seq(texts)
    classes = np.unique(labels)
    cw = compute_class_weight("balanced", classes=classes, y=labels)
    cw = {int(c): float(w) for c, w in zip(classes, cw)}
    model = build_bilstm(K)
    model.fit(
        Xs, labels, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.1,
        class_weight=cw, verbose=0,
        callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
    )
    return model

def predict_bilstm(model, texts):
    return model.predict(to_seq(texts), verbose=0)

## 7. Stacking (out-of-fold)

Out-of-fold predictions from the three base models are used as features for the meta-learner
(Logistic Regression), which avoids data leakage.

In [ ]:
from sklearn.linear_model import LogisticRegression

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
n_train = len(X_train)
oof = {name: np.zeros((n_train, K)) for name in ["nb", "svm", "bilstm"]}

for fold, (tr, va) in enumerate(skf.split(X_train, y_train), 1):
    print(f"Fold {fold}/{N_FOLDS}")
    Xtr, Xva, ytr = X_train[tr], X_train[va], y_train[tr]

    nb = make_nb().fit(Xtr, ytr);   oof["nb"][va] = nb.predict_proba(Xva)
    sv = make_svm().fit(Xtr, ytr);  oof["svm"][va] = sv.predict_proba(Xva)
    bl = train_bilstm(Xtr, ytr);    oof["bilstm"][va] = predict_bilstm(bl, Xva)
    tf.keras.backend.clear_session()

meta_train = np.hstack([oof["nb"], oof["svm"], oof["bilstm"]])

meta = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
meta.fit(meta_train, y_train)
print("Meta-learner trained.")

In [ ]:
# Retrain the base models on the full training set, then predict the test set
nb_full = make_nb().fit(X_train, y_train)
svm_full = make_svm().fit(X_train, y_train)
bilstm_full = train_bilstm(X_train, y_train)

p_nb = nb_full.predict_proba(X_test)
p_svm = svm_full.predict_proba(X_test)
p_bil = predict_bilstm(bilstm_full, X_test)

meta_test = np.hstack([p_nb, p_svm, p_bil])
y_pred = meta.predict(meta_test)       # <-- single final output of the ensemble

## 8. Evaluation of the ensemble model

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

print("=== STACKING ENSEMBLE (NB + SVM + BiLSTM) ===")
print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
print("F1-macro :", round(f1_score(y_test, y_pred, average="macro"), 4))
print()
print(classification_report(y_test, y_pred, target_names=le.classes_, digits=4))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix - Stacking")
plt.tight_layout(); plt.show()

In [ ]:
# (Optional) Supporting/ablation table: each base model vs the ensemble
rows = []
for name, p in [("Naive Bayes", p_nb), ("SVM", p_svm), ("BiLSTM", p_bil)]:
    pr = p.argmax(1)
    rows.append([name, accuracy_score(y_test, pr), f1_score(y_test, pr, average="macro")])
rows.append(["Stacking (NB+SVM+BiLSTM)", accuracy_score(y_test, y_pred), f1_score(y_test, y_pred, average="macro")])
pd.DataFrame(rows, columns=["Model", "Accuracy", "F1-macro"]).round(4)

## 9. Predict new comments

In [ ]:
def predict_comment(text):
    c = clean_text(text)
    f = np.hstack([
        nb_full.predict_proba([c]),
        svm_full.predict_proba([c]),
        predict_bilstm(bilstm_full, [c]),
    ])
    prob = meta.predict_proba(f)[0]
    return le.classes_[prob.argmax()], dict(zip(le.classes_, prob.round(3)))

# Example comments (Indonesian, like the dataset)
for s in [
    "IHSG hijau terus, mantap cuan hari ini",
    "IHSG merah lagi, mending cash dulu",
    "Besok IHSG dibuka jam berapa ya?",
]:
    print(s, "->", predict_comment(s))